# Twitch Analytics — Les facteurs de variation des audiences et l’effet GTA VI
**Projet fil rouge · API Twitch Helix · 7 jours · GitHub Actions**

**Problématique :** *Quels facteurs sont associés aux variations d’audience des streamers populaires sur Twitch, et comment l’arrivée de GTA VI modifie-t-elle les tendances observées ?*

**Protocole :** un relevé toutes les **4 heures**, jusqu’à **5 000 streams** de **1 000 spectateurs minimum**. Une ligne correspond à **un stream observé à une date donnée**, pas à un spectateur unique.

⚠️ Le fichier couvre uniquement les streams visibles dans ce périmètre. Les analyses montrent des **associations**, pas des causes démontrées. Les chiffres GTA VI affichés proviendront **uniquement de ton CSV**, jamais de données inventées.

## 1. Chargement et préparation des données

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

FICHIER = Path('twitch_helix_7jours.csv')
if not FICHIER.exists():
    raise FileNotFoundError('Place twitch_helix_7jours.csv dans le même dossier que ce notebook.')

brut = pd.read_csv(FICHIER, low_memory=False, encoding='utf-8-sig')
print(f'Fichier : {FICHIER.name} — {len(brut):,} lignes, {len(brut.columns)} colonnes'.replace(',', ' '))
display(brut.head(5))

# Compatibilité avec des relevés collectés avant l’enrichissement
df = brut.copy()
if 'stream_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id':'stream_id'})
if 'scheduled_at' not in df.columns:
    df['scheduled_at'] = df.get('collected_at')
for col in ['follower_count','follower_status','broadcaster_type',
            'stream_duration_minutes','viewer_follower_ratio','game_name',
            'user_id','user_name','stream_id','started_at','language']:
    if col not in df.columns: df[col] = np.nan
for col in ['collected_at','scheduled_at','started_at']:
    if col not in df.columns: df[col] = pd.NaT
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
for col in ['viewer_count','follower_count','stream_duration_minutes']:
    df[col] = pd.to_numeric(df.get(col), errors='coerce')
df['game_name'] = df['game_name'].fillna('Catégorie inconnue').astype(str).str.strip()
df['broadcaster_type'] = df['broadcaster_type'].fillna('inconnu').replace('', 'inconnu')
df['slot'] = df['scheduled_at'].fillna(df['collected_at'])
df['heure_paris'] = df['slot'].dt.tz_convert('Europe/Paris').dt.hour
df['jour_paris'] = df['slot'].dt.tz_convert('Europe/Paris').dt.strftime('%A')
if df['stream_duration_minutes'].notna().sum() == 0:
    df['stream_duration_minutes'] = (df['collected_at'] - df['started_at']).dt.total_seconds() / 60

# Normalisation ciblée des catégories, en gardant GTA V et GTA VI distincts
nom = df['game_name'].str.casefold()
df['gta_categorie'] = np.select([
    nom.str.contains(r'grand theft auto vi|grand theft auto 6|^gta vi$|^gta 6$', regex=True),
    nom.str.contains(r'grand theft auto v(?:\b|$)|grand theft auto 5|^gta v$|^gta 5$', regex=True)
], ['GTA VI','GTA V'], default='Autres')

invalides = (df['viewer_count'].isna() | (df['viewer_count'] < 1000) | df['slot'].isna())
df = df.loc[~invalides].copy()
# 1 stream ne doit apparaître qu’une fois par créneau
nb_doublons = int(df.duplicated(['slot','stream_id']).sum())
df = df.drop_duplicates(['slot','stream_id'])

plt.rcParams.update({'figure.figsize':(10,4.7),'axes.grid':True,'grid.alpha':0.18})
fmt = FuncFormatter(lambda x, pos: f'{x:,.0f}'.replace(',', ' '))
def fini(ax, axe='y'):
    (ax.yaxis if axe == 'y' else ax.xaxis).set_major_formatter(fmt)
    plt.tight_layout(); plt.show()
def indisponible(raison):
    print('Graphique indisponible : ' + raison)


## 2. Qualité et représentativité de l’échantillon
Vérifier les observations manquantes, la couverture temporelle et le nombre de followers réellement accessibles. **Ne jamais remplacer des followers manquants par zéro.**

In [ ]:
qualite = pd.DataFrame({
    'Valeurs manquantes': brut.isna().sum(),
    '% manquants': (brut.isna().mean()*100).round(1)
}).sort_values('% manquants',ascending=False)
display(qualite)

bilan = pd.Series({
    'Observations exploitables':len(df),
    'Relevés distincts':df['slot'].nunique(),
    'Streamers distincts':df['user_id'].nunique(),
    'Catégories distinctes':df['game_name'].nunique(),
    'Doublons supprimés':nb_doublons,
    'Audience médiane':df['viewer_count'].median() if len(df) else np.nan,
    'Followers disponibles (%)':round(df['follower_count'].notna().mean()*100,1) if len(df) else 0,
    'Début (UTC)':df['slot'].min(),
    'Fin (UTC)':df['slot'].max(),
})
display(bilan.to_frame('Résultat'))
if df['follower_count'].notna().sum()==0:
    print('ATTENTION : aucun nombre de followers accessible avec les autorisations courantes ; la notoriété ne peut pas encore être testée.')
if df['slot'].nunique()<2:
    print('ATTENTION : un seul relevé. Les analyses de variation temporelle attendent plusieurs collectes.')


## 3. Figure 1 — Distribution des audiences (zoom au 95e percentile)
**Question :** les audiences sont-elles concentrées sur quelques grands streamers ? Les valeurs extrêmes restent dans le CSV et dans les statistiques.

In [ ]:
if df.empty:
    indisponible('aucune observation valide')
else:
    plafond = df['viewer_count'].quantile(.95)
    valeurs = df.loc[df['viewer_count'] <= plafond,'viewer_count']
    fig,ax=plt.subplots()
    ax.hist(valeurs, bins=32,edgecolor='white')
    ax.set(xlabel='Spectateurs par stream',ylabel='Nombre d’observations',
           title='Répartition des audiences — 95 % des valeurs les plus basses')
    fini(ax,'x')
    print(f'95e percentile : {plafond:,.0f} spectateurs ; maximum réel : {df.viewer_count.max():,.0f}'.replace(',',' '))


## 4. Figure 2 — Nombre de followers et audience
**Hypothèse :** les streamers avec davantage de followers attirent-ils davantage de spectateurs ? Un nuage de points permet de voir la relation, sans en déduire automatiquement une causalité.

Si les followers sont absents parce que l’API n’a pas donné les permissions, le graphique ne sera pas créé.

In [ ]:
notoriete = df.dropna(subset=['follower_count','viewer_count']).query('follower_count > 0').copy()
if len(notoriete) < 10:
    indisponible(f'followers accessibles pour seulement {len(notoriete)} observations (10 minimum)')
else:
    # 1 point par streamer : médianes pour éviter de surpondérer les streamers souvent observés
    chaines = notoriete.groupby('user_id').agg(followers=('follower_count','median'),
                        audience=('viewer_count','median')).dropna()
    if len(chaines)<5:
        indisponible('moins de 5 chaînes ayant un total de followers')
    else:
        fig,ax=plt.subplots()
        ax.scatter(chaines.followers, chaines.audience, alpha=.5, s=25)
        ax.set(xlabel='Followers (médiane par chaîne)',ylabel='Spectateurs (médiane par chaîne)',
               title='Notoriété et audience observée')
        fini(ax,'x')
        print(f'Chaînes étudiées : {len(chaines)} ; corrélation de Spearman : {chaines.followers.corr(chaines.audience,method="spearman"):.2f}')


## 5. Figure 3 — Statut du streamer et audience
**Question :** les partenaires Twitch sont-ils associés à des audiences médianes plus fortes que les affiliés ? Attention aux groupes peu représentés.

In [ ]:
val = df[df['broadcaster_type'].isin(['partner','affiliate'])]
if val.empty:
    indisponible('statut partner/affiliate absent')
else:
    med = val.groupby('broadcaster_type')['viewer_count'].median().sort_values()
    fig,ax=plt.subplots(figsize=(8,4))
    ax.barh(med.index,med.values)
    ax.set(xlabel='Audience médiane',title='Audience médiane selon le statut Twitch')
    fini(ax,'x')
    display(val.groupby('broadcaster_type').agg(observations=('viewer_count','size'),
                    streamers=('user_id','nunique')))


## 6. Figure 4 — Durée du live et audience
**Question :** les lives plus longs ont-ils une audience différente ? Zoom sur les audiences jusqu’au 95e percentile pour rendre le graphique lisible.

In [ ]:
dur = df.dropna(subset=['stream_duration_minutes','viewer_count'])
dur = dur[dur['stream_duration_minutes'].between(0,24*60)]
if len(dur)<10:
    indisponible('durée de live insuffisante')
else:
    p95=dur.viewer_count.quantile(.95)
    fig,ax=plt.subplots()
    ax.scatter(dur.stream_duration_minutes/60,dur.viewer_count.clip(upper=p95),s=9,alpha=.2)
    ax.set(xlabel='Durée du live au relevé (heures)',ylabel='Spectateurs (plafonnés au P95 pour affichage)',
           title='Durée de diffusion et audience observée')
    fini(ax)
    print('Le plafonnement est uniquement visuel : toutes les audiences sont conservées dans les calculs.')


## 7. Figure 5 — Variations d’audience selon les heures
**Question :** quels créneaux de collecte sont associés aux meilleures audiences ? Les horaires sont convertis en heure de Paris. Avec une collecte toutes les 4 heures, seuls quelques créneaux seront représentés.

In [ ]:
if df['slot'].nunique()<2:
    indisponible('au moins deux relevés nécessaires')
else:
    stats = df.groupby('heure_paris')['viewer_count'].median().sort_index()
    fig,ax=plt.subplots()
    ax.plot(stats.index,stats.values,marker='o')
    ax.set(xticks=stats.index,xlabel='Heure de collecte (Paris)',ylabel='Audience médiane',
           title='Audience observée selon le créneau horaire')
    fini(ax)


## 8. Figure 6 — GTA V et GTA VI : audiences observées par relevé
**Contexte :** Rockstar annonce la sortie de **GTA VI le 19 novembre 2026**. On étudie si les catégories GTA évoluent pendant la période de collecte.

**Précautions :** cette analyse ne couvre que les streams sélectionnés (≥ 1 000 spectateurs, maximum 2 000 par relevé). Elle ne représente **pas l’audience totale de Twitch**. Pour isoler un effet avant/après la sortie, il faut réellement disposer de relevés des deux périodes.

In [ ]:
gta=df[df.gta_categorie.isin(['GTA V','GTA VI'])]
if gta.empty:
    indisponible('aucune catégorie GTA V ou GTA VI présente dans les relevés')
else:
    serie=gta.pivot_table(index='slot',columns='gta_categorie',values='viewer_count',aggfunc='sum',fill_value=0)
    for nom in ['GTA V','GTA VI']:
        if nom not in serie: serie[nom]=0
    fig,ax=plt.subplots(figsize=(11,4.5))
    for nom in ['GTA V','GTA VI']:
        ax.plot(serie.index,serie[nom],marker='o',label=nom)
    sortie=pd.Timestamp('2026-11-19',tz='Europe/Paris').tz_convert('UTC')
    if serie.index.min() <= sortie <= serie.index.max():
        ax.axvline(sortie,linestyle='--',color='gray',label='19 nov. (date de sortie)')
    ax.set(xlabel='Date de relevé (UTC)',ylabel='Somme des spectateurs observés',
           title='GTA V et GTA VI — audiences dans notre échantillon')
    ax.legend();fini(ax)
    if len(serie)<2:print('Un seul relevé : évolution impossible à déterminer.')


## 9. Figure 7 — Part de GTA VI dans les audiences observées
**Question :** quelle proportion de l’audience des streams sélectionnés est associée à GTA VI ? Cette part est calculée **parmi les streams retenus**, et non sur la plateforme entière.

In [ ]:
if df['slot'].nunique()<2 or not (df['gta_categorie']=='GTA VI').any():
    indisponible('plusieurs relevés et au moins un stream GTA VI nécessaires')
else:
    obs=df.groupby('slot')['viewer_count'].sum()
    vi=df[df['gta_categorie']=='GTA VI'].groupby('slot')['viewer_count'].sum()
    part=(100*vi.reindex(obs.index,fill_value=0)/obs).sort_index()
    fig,ax=plt.subplots()
    ax.plot(part.index,part.values,marker='o')
    ax.set(xlabel='Relevé (UTC)',ylabel='Part de l’audience observée (%)',
           ylim=(0,max(5,min(100,float(part.max())*1.15))),
           title='Part de GTA VI parmi les streams populaires suivis')
    plt.tight_layout();plt.show()


## 10. Figure 8 — GTA VI comparé aux autres grandes catégories
**Question :** GTA VI prend-il une place importante face aux autres jeux ? On compare la part des spectateurs par catégorie, sur les mêmes relevés. La somme sur plusieurs relevés n’est **pas** un total de spectateurs uniques.

In [ ]:
if df.empty:
    indisponible('aucune donnée')
else:
    top=(df.groupby('game_name')['viewer_count'].sum().nlargest(10).sort_values())
    fig,ax=plt.subplots(figsize=(10,5.2))
    ax.barh(top.index,top.values)
    ax.set(xlabel='Somme des audiences des relevés (pas de spectateurs uniques)',
           title='10 catégories les plus présentes en audience observée')
    fini(ax,'x')


## 11. Figure 9 — Streamers passant de GTA V à GTA VI
**Question :** parmi les chaînes suivies plusieurs fois, combien sont observées sur GTA V puis sur GTA VI ? Cela décrit des changements de catégorie, sans prouver que GTA VI a causé leur changement d’audience.

In [ ]:
gta=df[df.gta_categorie.isin(['GTA V','GTA VI'])].sort_values('slot')
changements=[]
for uid,sub in gta.groupby('user_id'):
    cats=sub['gta_categorie'].tolist()
    if 'GTA V' in cats and 'GTA VI' in cats:
        t_v=sub.loc[sub.gta_categorie=='GTA V','slot'].min()
        t_vi=sub.loc[sub.gta_categorie=='GTA VI','slot'].max()
        if t_v<t_vi:
            changements.append((uid,sub.user_name.iloc[0],
                sub.loc[sub.gta_categorie=='GTA V','viewer_count'].median(),
                sub.loc[sub.gta_categorie=='GTA VI','viewer_count'].median()))
if not changements:
    indisponible('aucun passage GTA V → GTA VI observable dans la période')
else:
    passage=pd.DataFrame(changements,columns=['user_id','streamer','mediane_GTA_V','mediane_GTA_VI'])
    print(f'Streamers observés passant de GTA V à GTA VI : {len(passage)}')
    top=passage.sort_values('mediane_GTA_VI',ascending=False).head(10)
    fig,ax=plt.subplots(figsize=(10,5))
    x=np.arange(len(top))
    ax.bar(x-.19,top.mediane_GTA_V,width=.38,label='GTA V')
    ax.bar(x+.19,top.mediane_GTA_VI,width=.38,label='GTA VI')
    ax.set_xticks(x,top.streamer,rotation=35,ha='right')
    ax.set(ylabel='Audience médiane observée',title='Mêmes streamers : audiences observées selon le jeu')
    ax.legend();fini(ax)


## 12. Bilan pour ton oral

- **Objectif :** expliquer les différences et les variations d’audience des streams populaires, à partir de facteurs mesurables.
- **Variables :** followers (si autorisés), statut, jeu, heure, jour, durée de live et nombre de spectateurs.
- **Actualité GTA VI :** étudier si GTA VI gagne en visibilité par rapport à GTA V et aux autres catégories durant les relevés.
- **Qualité :** contrôler les doublons, les dates, les valeurs manquantes et les autorisations pour les followers.
- **Limites :** le seuil de 1 000 spectateurs et les 2 000 streams maximum créent un biais de sélection ; une semaine et un pas de 4 h peuvent manquer des pics. Un changement de tendance ne prouve pas une causalité.
- **Suite :** constituer plusieurs semaines de relevés, entraîner une première prédiction d’audience sur les données passées, puis la tester sur des dates ultérieures (sans mélanger passé et futur).

**Attention spécifique à GTA VI :** si ta collecte commence le 19 novembre 2026, tu n’auras pratiquement pas de relevés pré-sortie. Tu pourras montrer l’évolution **pendant la semaine de lancement**, mais pas mesurer solidement un effet **avant/après**.

**Source événementielle :** [Rockstar Games — sortie GTA VI le 19 novembre 2026](https://www.rockstargames.com/newswire/article/ak3ak31a49a221/grand-theft-auto-vi-is-now-set-to-launch-november-19-2026).